<a target="_blank" href="https://colab.research.google.com/github/novainsilico/jinko-api-cookbook/blob/main/tutorials/03-post-calibration-uq.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Post-calibration uncertainty quantification

### Introduction
The goal of this tutorial is to quantify the uncertainty around the result of an existing calibration.
It takes a completed calibration as input and explores the parameter space around its best patient (the maximum likelihood estimate, or MLE).
It outputs an exploratory Vpop and an exploratory Trial in the folder of your choice, and plots the likelihood landscape within the notebook.

### Steps:
0. Connect to jinkō and set up the tutorial
1. Load the calibration of interest
2. Get the maximum likelihood estimate (MLE)
3. Generate an exploratory Vpop
4. Create or update the exploratory Trial
5. Run and monitor the Trial
6. Load and process the Trial results
7. Plot the uncertainty quantification report
8. Approximate the Hessian matrix of the likelihood
9. Analyze the quality of the approximation

### Linked resources:
- [API tutorials](https://doc.jinko.ai/docs/category/api---tutorials/): guidelines and examples on how to use the SDK
- [API Reference](https://doc.jinko.ai/api/#/): exhaustive documentation of jinkō's API
- [Public cookbook repository](https://github.com/novainsilico/jinko-api-cookbook/tree/main): cookbooks examples of use cases we compiled
- [jinko.ai](https://jinko.ai): jinkō website, you'll need an account

### What you'll need
- A Jinko API key with access to your project.
- The project id (UUID).
- A calibration that has already run in this project, and its short id.
- A folder id in this project, where the exploratory Vpop and Trial will be saved.
- Python environment: [jinko-sdk](https://pypi.org/project/jinko-sdk/) package installed, along with `numpy`, `pandas`, `plotly`, `scipy` and `scikit-learn`. In Colab, `jinko-sdk` is installed via the notebook and the other packages are available by default.
- Optional: set `JINKO_API_KEY` and `JINKO_PROJECT_ID` via `.envrc` (see `.envrc.sample`).

---

## Step 0 : Connect to jinkō and set up the tutorial

In [ ]:
import os
import sys
import textwrap
import uuid

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import qmc
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, minmax_scale

In [ ]:
if "google.colab" in sys.modules:
    !pip install -q jinko-sdk

from jinko import JinkoClient

### Connection initialization
- See the [Getting Started](https://github.com/novainsilico/jinko-api-cookbook/blob/main/tutorials/01-getting_started.ipynb) notebook for fuller setup instructions.
- Create an API key for your project (project settings → API Access) and store it in your secrets (`.envrc` locally or Colab userdata).
- The project id is the UUID in the project URL.

In [ ]:
# Load secrets from Colab (if running in Colab)
if "google.colab" in sys.modules:
    from google.colab import userdata

    api_key = userdata.get("JINKO_API_KEY")
    project_id = userdata.get("JINKO_PROJECT_ID")
    if api_key:
        os.environ["JINKO_API_KEY"] = api_key
        print("JINKO_API_KEY is set")
    if project_id:
        os.environ["JINKO_PROJECT_ID"] = project_id
        print("JINKO_PROJECT_ID is set")
else:
    api_key = None
    project_id = None

client = JinkoClient(api_key=api_key, project_id=project_id)
client.auth_check()

constants of the tutorial:

In [ ]:
# CALIBRATION_SID can be retrieved in the url of your calibration, pattern is `https://jinko.ai/<calibration_short_id>`
CALIBRATION_SID = ""  # TO BE PROVIDED

# FOLDER_ID can be retrieved in the url of the folder, pattern is `https://jinko.ai/project/<project_id>?labels=<folder_id>`
# The exploratory Vpop and Trial will be saved in this folder. It does not have to be the folder of the calibration.
FOLDER_ID = ""  # TO BE PROVIDED

if not CALIBRATION_SID or not FOLDER_ID:
    raise ValueError(
        "Set CALIBRATION_SID and FOLDER_ID before running the rest of the notebook.")

# Step 1 : Load the calibration of interest

By default, we use the latest version of the calibration.
You can pick another version by setting either its revision number or its label.

**Warning:** we cache the exploratory Trial so that re-running the notebook creates new versions of the same Trial instead of new Trials.
When you change `CALIBRATION_SID`, restart the kernel to clear this cache.

In [ ]:
# Choose a specific revision. Leave it to None to use the latest version
CALIBRATION_REVISION = None
# Choose a specific label. Leave it to None to use `CALIBRATION_REVISION` instead
CALIBRATION_LABEL = None

revision = CALIBRATION_REVISION
if CALIBRATION_LABEL:
    revision = next(
        (
            version.revision
            for version in client.get_calibration(CALIBRATION_SID).versions.iter(only_labeled=True)
            if version.label == CALIBRATION_LABEL
        ),
        None,
    )
    if revision is None:
        raise ValueError(
            f"No version labeled {CALIBRATION_LABEL!r} on calibration {CALIBRATION_SID}")
calibration = client.get_calibration(CALIBRATION_SID, revision=revision)

print(
    f"Picked Calibration with coreItemId: {calibration.core_id}, snapshotId: {calibration.snapshot_id}")
print(f"Resource link: {calibration.url}")

calibration_status = calibration.status().get("status")
print(f"Calibration status: {calibration_status}")
if calibration_status not in {"completed", "stopped"}:
    print("Warning: this calibration version has not finished, its results may be missing or partial.")

calibration_inputs = calibration.content(revision=revision)
parameters = [p["id"] for p in calibration_inputs["parameters"]]
parameter_map = {p["id"]: p for p in calibration_inputs["parameters"]}
print(f"{len(parameters)} calibrated parameters:")
display(parameters)

# Step 2 : Get the maximum likelihood estimate (MLE)

The MLE is the best patient of the calibration, in other words the patient with the highest `optimizationWeightedScore`.

In [ ]:
arm_names = calibration.results_summary()["arms"]

sorted_patients = calibration.results.sorted_patients(
    sort_by="optimizationWeightedScore")
best_patient_index = sorted_patients[0]

# Get the list of scalars that we want to download, they correspond to the calibration objectives
objectives = calibration.objective_weights()

arm_names_with_cross_arms = [*arm_names, "crossArms"]
best_patient_raw = calibration.results.scalars_per_patient(
    patient_number=best_patient_index["patientNumber"],
    scalars=[],  # all scalars
    arms=arm_names_with_cross_arms,
    iteration=int(best_patient_index["iteration"]),
)
cross_arm_values = list(
    filter(
        lambda x: x["scenarioArm"] == "crossArms",
        best_patient_raw["outputs"],
    )
)
if len(cross_arm_values) == 0:
    raise ValueError(f"Expected at least one cross-arm value")
best_patient_values = cross_arm_values[0]["res"]
best_patient_parameters = {
    p["id"]: float(p["value"])
    for p in best_patient_values
    if p["type"]["type"] == "BaselineInput"
}
best_patient_scores = {
    s["id"]: s["value"]
    for results in best_patient_raw["outputs"]
    for s in results["res"]
    if (s["id"] in objectives.keys() or s["id"] == "optimizationWeightedScore")
}
mle = best_patient_scores["optimizationWeightedScore"]
print(
    f"Best patient corresponds to a maximum log-likelihood of {mle:.2g} (theoretical maximum is 1)"
)
print("All scores of the best patient:")
display(best_patient_scores)
best_patient_scores_trans = {
    k: np.exp(best_patient_scores[k] - 1) for k in best_patient_scores
}
print("Best patient descriptors:")
display(best_patient_parameters)

# Step 3 : Generate an exploratory Vpop

We generate the exploratory Vpop directly from a list of patients, without going through a Vpop Design.
It merges two sub-populations:
- a "multi" sub-population, where all parameters vary simultaneously (Sobol sequence over the exploration hypercube);
- one "tensorized" sub-population per parameter, where only that parameter varies while the others stay at their MLE value.

By default, each parameter is explored between half and twice its MLE value, within the calibration bounds.
Use `CUSTOM_RANGE` to override the exploration range or the log scale of specific parameters.

In [ ]:
def to_patient(x):
    return {
        "patientIndex": str(uuid.uuid4()),
        "patientCategoricalAttributes": [],
        "patientAttributes": [{"id": p, "val": x[j]} for j, p in enumerate(parameters)],
    }


# Defines the low and high bounds to explore for each parameter which is not in the "CUSTOM_RANGE" map
low_multiply = 1 / 2  # multiplies the MLE value to get the low bound
high_multiply = 2  # multiplies the MLE value to get the high bound

# Set custom ranges and logscale here. Simply add a new entry for each parameter that needs to be tweaked
# Note that "min" and "max" refer to the values of "x" (not "log(x)"), regardless of the value of "logTransform",
# just like in Jinko.ai. Any key left out falls back to the default behavior.
# Example:
# CUSTOM_RANGE = {
#     "kClearanceDrug": {"logTransform": True, "min": 1e-3, "max": 1e0},
#     "bloodFlowRate": {"logTransform": False, "min": 1, "max": 50},
# }
CUSTOM_RANGE = {}

unknown_parameters = set(CUSTOM_RANGE) - set(parameters)
if unknown_parameters:
    raise ValueError(
        f"CUSTOM_RANGE has parameters that are not calibrated: {sorted(unknown_parameters)}")


def parameter_log_scaled(param_name):
    p = parameter_map[param_name]
    if param_name in CUSTOM_RANGE and "logTransform" in CUSTOM_RANGE[param_name]:
        return CUSTOM_RANGE[param_name]["logTransform"]
    elif p.get("logTransform") == True:
        return True
    else:
        return False


def parameter_low_bound(param_name):
    p = parameter_map[param_name]
    if CUSTOM_RANGE.get(param_name, {}).get("min") is not None:
        min_x = CUSTOM_RANGE[param_name]["min"]
    elif p["minBound"] is not None:
        min_x = max(
            best_patient_parameters[f"{param_name}.tmin"] *
            low_multiply, p["minBound"]
        )
    else:
        min_x = best_patient_parameters[f"{param_name}.tmin"] * low_multiply
    if parameter_log_scaled(param_name):
        return np.log10(min_x)
    else:
        return min_x


def parameter_high_bound(param_name):
    p = parameter_map[param_name]
    if CUSTOM_RANGE.get(param_name, {}).get("max") is not None:
        max_x = CUSTOM_RANGE[param_name]["max"]
    elif p["maxBound"] is not None:
        max_x = min(
            best_patient_parameters[f"{param_name}.tmin"] *
            high_multiply, p["maxBound"]
        )
    else:
        max_x = best_patient_parameters[f"{param_name}.tmin"] * high_multiply
    if parameter_log_scaled(param_name):
        return np.log10(max_x)
    else:
        return max_x


"""
First we generate an exploratory Vpop using Sobol sequences to sample from the exploration hypercube
around the MLE parameter values
"""
# The size of the exploratory Vpop is 2^m, e.g. vpop size is 1024 for m=10
m = 10

d = len(parameters)
sampler = qmc.Sobol(d, scramble=False)
samples = sampler.random_base2(m)
samples = qmc.scale(
    samples,
    [parameter_low_bound(param_name) for param_name in parameters],
    [parameter_high_bound(param_name) for param_name in parameters],
)
# Handle log scaled parameters
for j, param_name in enumerate(parameters):
    if parameter_log_scaled(param_name):
        samples[:, j] = np.exp(np.log(10) * samples[:, j])

exploratory_vpop = [to_patient(x) for x in samples]

"""
Then we generate a "tensorized" Vpop where, for each parameter p_i, we linearly sample on the exploration
interval while keeping p_j,j/=i fixed to the MLE values
"""
# how many samples in the search interval
n = 21

patient_groups = {}
tensor_vpop = []
for param_name in parameters:
    param_vals = np.linspace(
        parameter_low_bound(param_name), parameter_high_bound(param_name), n
    )
    if parameter_log_scaled(param_name):
        param_vals = np.exp(np.log(10) * param_vals)
    all_vals = [
        [
            x if p == param_name else best_patient_parameters[f"{p}.tmin"]
            for p in parameters
        ]
        for x in param_vals
    ]
    patients = [to_patient(x) for x in all_vals]
    patient_groups[param_name] = set(
        patient["patientIndex"] for patient in patients)
    tensor_vpop += patients

"""
We assemble a map from patient_id to a 'label' which is either one of the parameters IDs in which case the
patient belongs to the 1-D grid associated with that parameter. Or 'label' is 'multi' in which case the patient belongs
to the "exploratory" Vpop where all parameters vary simultaneously.
"""
patient_map = {}
for param_name, id_set in patient_groups.items():
    for patient_id in id_set:
        patient_map[patient_id] = param_name
for patient in exploratory_vpop:
    patient_map[patient["patientIndex"]] = "multi"

# We merge both Vpops, we will later use 'patient_map' to split the Vpop results
all_patients = tensor_vpop + exploratory_vpop
vpop = client.create_vpop_from_patients_list(
    all_patients, name=f"UQ Vpop for Calibration {CALIBRATION_SID}", folder=FOLDER_ID
)

print(f"Generated a Vpop of {len(all_patients)} patients")
print(f"Resource link: {vpop.url}")

# Step 4 : Create or update the exploratory Trial

The Trial reuses the model, protocol, data tables, solving options and scoring design of the calibration.

In [ ]:
trial_data = {
    "computationalModelId": calibration_inputs["computationalModelId"],
    "vpopId": {"coreItemId": vpop.core_id, "snapshotId": vpop.snapshot_id},
    "dataTableDesigns": calibration_inputs["dataTableDesigns"],
    "solvingOptions": calibration_inputs["solvingOptions"],
    "scoringDesignId": calibration_inputs["scoringDesignId"],
}

# If there's already a Trial in the cache, create a new version of the same Trial instead of creating a new one
# Restart the notebook kernel when changing calibrations.
if "trial" in globals():
    # SDK gap: no public method to update a trial's full payload in place.
    client.raw_request(
        "PATCH",
        f"/core/v2/trial_manager/trial/{trial.core_id}/snapshots/{trial.snapshot_id}",
        json_body=trial_data,
    )
    trial = client.get_trial(trial.sid)
else:
    trial = client.create_trial_from_json(
        json_content=trial_data,
        name=f"UQ for Calibration {CALIBRATION_SID}",
        folder=FOLDER_ID,
    )

print(f"Resource link: {trial.url}")

# Step 5 : Run and monitor the Trial

In [ ]:
trial.run()
final_status = trial.wait_until_completed()
print("Per-arm summary:")
for arm, counts in final_status.get("perArmSummary", {}).items():
    print(f"  {arm}: {counts}")

# Step 6 : Load and process the Trial results

In [ ]:
scalars_summary = trial.results.summary()
print("Number of patients:", len(scalars_summary["patients"]))
print("Available arms:", scalars_summary["arms"])
print("Available scalars:\n", [s["id"] for s in scalars_summary["scalars"]])
print("Available cross-arm scalars:\n", [s["id"]
      for s in scalars_summary["scalarsCrossArm"]])

# Find baseline inputs, in other words the patients descriptors which were calibrated
patientDescriptors = [f"{p}.tmin" for p in parameters]

print("List of patient descriptors:\n", patientDescriptors, "\n")

resultScalars = list(objectives.keys()) + ["optimizationWeightedScore"]

print("List of result scalars:\n", resultScalars, "\n")

In [ ]:
dfScalars = trial.results.scalars(resultScalars).to_dataframe()
dfDescriptors = trial.results.scalars(patientDescriptors).to_dataframe()

In [ ]:
"""
Filter the results by keeping only patients whose 'optimizationWeightedScore' is GREATER THAN a given threshold
"""

weightedScoreThreshold = -1e33


# Renaming and filtering some columns
dfScalars = dfScalars.rename(columns={"value": "likelihood", "scalarId": "score"}).drop(
    columns=["unit", "armId"]
)
# handle scalar errors
dfScalars["likelihood"] = pd.to_numeric(
    dfScalars["likelihood"], downcast="float", errors="coerce"
)

print("\nRaw scalar results data (first rows):\n")
display(dfScalars.head())

# Renaming and filtering
dfDescriptors = dfDescriptors.rename(
    columns={"value": "parameterValue", "scalarId": "parameterName"}
).drop(columns=["unit", "armId"])
# Remove the `.tmin` suffix to parameter names
dfDescriptors["parameterName"] = dfDescriptors["parameterName"].apply(
    lambda t: t.removesuffix(".tmin")
)
print("\nPatients descriptors (first rows):\n")
display(dfDescriptors.head())

# Filter on optimizationWeightedScore
dfPatients = dfScalars.loc[
    (dfScalars["score"] == "optimizationWeightedScore")
    & (dfScalars["likelihood"] > weightedScoreThreshold)
]["patientId"]
print(f"\nNumber of selected patients: {len(dfPatients)}")

# Final data set
# Merge the scalars and descriptors
dfFull = dfScalars.merge(dfDescriptors, how="inner", on=["patientId"])
# Filter the data set
dfFull = dfFull.loc[dfFull["patientId"].isin(dfPatients)]
# Apply a transformation to likelihood
dfFull["likelihoodTrans"] = dfFull["likelihood"].apply(lambda y: np.exp(y - 1))

# Label the rows depending on whether they belong to the "tensorized" sub-vpop or
# the "exploratory" sub-vpop
dfFull["label"] = dfFull["patientId"].apply(
    lambda patient_id: patient_map[patient_id])

print("\nMerged data set first 5 rows:\n")
display(dfFull.head())

# Step 7 : Plot the uncertainty quantification report

This section generates a grid of individual scatter plots. Each plot shows an individual score value in the y-axis against an individual parameter value in the x-axis.

The rows of the grid correspond to different scores used in the calibration, with the first row being the `optimizationWeightedScore`. By default only this row is shown, set `SHOW_ALL_SCORES = True` to display all the calibration scores.

The columns of the grid correspond to the different parameters in the calibration options. Log-scaled parameters are displayed with a log-scale.

Each plot contains

- A green-cross: this shows the position (parameter value - score value) of the best patient from the calibration
- A red-line: this shows the results of the single parameter sweep, for each considered parameter (depending on the facet). This is a very rough estimation of the likelihood profile in the direction of one parameter.
- A scatter plot of blue points: these are all the individual patients in the multi-dimensional exploration trial. This gives an idea of the shape of the likelihood landscape in multi-dimension. Keep in mind that this plot is affected by the curse of dimensionality, and the required number of patients to precisely estimate the value of the likelihood increases exponentially with the number of parameters.

Parameter identifiability is associated to the "sharpness" of the likelihood profile around the maximal value.

![Output plot explained](https://raw.githubusercontent.com/novainsilico/jinko-api-cookbook/main/tutorials/resources/post_calibration_uq/explo-calib-uq.png)

In [ ]:
# Set to True to add one row per calibration score, below the `optimizationWeightedScore` row
SHOW_ALL_SCORES = False

scoreList = ["optimizationWeightedScore"]
if SHOW_ALL_SCORES:
    scoreList += [s for s in dfFull.score.unique() if s !=
                  "optimizationWeightedScore"]
nbScores = len(scoreList)

scoreListWrapped = ["<br>".join(textwrap.wrap(t, width=12)) for t in scoreList]
display(scoreListWrapped)

nbParams = len(parameters)
parametersWrapped = ["<br>".join(textwrap.wrap(t, width=30))
                     for t in parameters]

# The figure grows with the number of parameters and scores
figureWidth = max(600, 250 * nbParams)
figureHeight = max(500, 250 * nbScores)

fig = make_subplots(
    len(scoreList),
    len(parameters),
    shared_xaxes=True,
    shared_yaxes=True,
    horizontal_spacing=0.01,
    vertical_spacing=0.01,
    column_titles=parametersWrapped,
    row_titles=scoreListWrapped,
)
for i in range(nbScores):
    for j in range(nbParams):
        df = dfFull.loc[
            (dfFull["parameterName"] == parameters[j])
            & (dfFull["score"] == scoreList[i])
        ]
        scatter_df = df.loc[df["label"] == "multi"]
        fig.add_trace(
            go.Scatter(
                x=scatter_df["parameterValue"],
                y=scatter_df["likelihoodTrans"],
                mode="markers",
                marker=dict(size=4, color="darkblue", opacity=0.5),
                hoverinfo="none",
            ),
            row=i + 1,
            col=j + 1,
        )
        mono_df = df.loc[df["label"] == parameters[j]
                         ].sort_values(by="parameterValue")
        fig.add_trace(
            go.Scatter(
                x=mono_df["parameterValue"],
                y=mono_df["likelihoodTrans"],
                line_width=3,
                line_color="red",
            ),
            row=i + 1,
            col=j + 1,
        )
        fig.add_trace(
            go.Scatter(
                x=[best_patient_parameters[f"{parameters[j]}.tmin"]],
                y=[best_patient_scores_trans[scoreList[i]]],
                mode="markers",
                marker=dict(
                    size=10,
                    color="white",
                    symbol="cross",
                    line=dict(color="green", width=1),
                ),
            ),
            row=i + 1,
            col=j + 1,
        )
        # special case for log-scaled parameters
        if parameter_log_scaled(parameters[j]):
            fig.update_xaxes(type="log", row=i + 1,
                             col=j + 1, tickformat=".1e")
fig.update_annotations(font_size=10)
fig.update_layout(
    font=dict(size=10),
    showlegend=False,
    width=figureWidth,
    height=figureHeight,
    template="plotly_white",
)
fig.show()

In [ ]:
# Additional plot: pairwise correlation plots for calibrated parameters

# Log-scaled parameters are displayed with a log scale
figureHeight = max(600, 150 * nbParams)
figureWidth = max(600, 150 * nbParams)

fig = make_subplots(
    len(parameters),
    len(parameters),
    shared_xaxes=True,
    shared_yaxes=True,
    horizontal_spacing=0.01,
    vertical_spacing=0.01,
    column_titles=parametersWrapped,
    row_titles=parametersWrapped,
)
for i in range(nbParams):  # iterating over rows
    for j in range(nbParams):  # iterating over columns
        if i == j:  # do not plot anything on the diagonal
            fig.add_trace(
                go.Scatter(x=[], y=[]),
                row=i + 1,
                col=j + 1,
            )
        else:
            df_j = dfFull.loc[
                (dfFull["parameterName"] == parameters[j])
                & (dfFull["score"] == "optimizationWeightedScore")
                & (dfFull["label"] == "multi")
            ]
            df_j = df_j[["parameterValue", "likelihoodTrans", "patientId"]].rename(
                columns={"parameterValue": parameters[j]}
            )
            df_i = dfFull.loc[
                (dfFull["parameterName"] == parameters[i])
                & (dfFull["score"] == "optimizationWeightedScore")
                & (dfFull["label"] == "multi")
            ]
            df_i = df_i[["parameterValue", "patientId"]].rename(
                columns={"parameterValue": parameters[i]}
            )

            scatter_df = df_j.merge(df_i, how="inner", on=["patientId"])
            scatter_df = scatter_df.sort_values(by="likelihoodTrans")
            fig.add_trace(
                go.Scatter(
                    # x axis is the column parameter
                    x=scatter_df[parameters[j]],
                    y=scatter_df[parameters[i]],  # y axis is the row parameter
                    mode="markers",
                    marker=dict(
                        size=6,
                        color=scatter_df["likelihoodTrans"],
                        opacity=(
                            scatter_df["likelihoodTrans"]
                            / scatter_df["likelihoodTrans"].max()
                        ),
                        coloraxis="coloraxis1",
                    ),
                    hoverinfo="none",
                ),
                row=i + 1,
                col=j + 1,
            )

        # Set the log scale for x and / or y axes
        if parameter_log_scaled(parameters[j]):
            fig.update_xaxes(type="log", row=i + 1,
                             col=j + 1, tickformat=".1e")
        if parameter_log_scaled(parameters[i]):
            fig.update_yaxes(type="log", row=i + 1,
                             col=j + 1, tickformat=".1e")
fig.update_annotations(font_size=10)
fig.update_xaxes(tickangle=40)
fig.update_coloraxes(
    colorbar_title="Likelihood",
    cmin=0,
    cmax=0.9,
    colorscale="Rainbow",
    colorbar_thickness=15,
)
fig.update_layout(
    font=dict(size=10),
    showlegend=False,
    # coloraxis_showscale=True,
    width=figureWidth,
    height=figureHeight,
    template="plotly_white",
)
fig.show()

# Step 8 : Approximate the Hessian matrix of the likelihood

For the "best patient" to be at a local maximum likelihood, all Hessian matrix eigenvalues must be negative and the gradient must be zero.
We approximate them by fitting a quadratic model to the likelihood of the "multi" sub-population.
See the [hessian_matrix](https://github.com/novainsilico/jinko-api-cookbook/blob/main/cookbooks/modeling/hessian_matrix.ipynb) cookbook for another example.

In [ ]:
# Massage the data such that we have a matrix of samples of size (N, dim) and a vector of likelihood values of size N
df = dfFull.loc[
    (dfFull["label"] == "multi") & (
        dfFull["score"] == "optimizationWeightedScore")
]
pivot_df = df.pivot(index="patientId",
                    columns="parameterName", values="parameterValue")

likelihood_df = df.groupby(by="patientId")["likelihood"].first().to_frame()
likelihood_df_sorted = likelihood_df.sort_values(
    by="patientId", ascending=False)
pivot_df_sorted = pivot_df.loc[likelihood_df_sorted.index]
unscaled_samples = pivot_df_sorted.values

# Scale the samples
# The centroid is the point in the paramter space corresponding to the MLE
unscaled_centroid = np.array(
    [
        best_patient_parameters[f"{param_name}.tmin"]
        for param_name in pivot_df_sorted.columns
    ]
)
unscaled_x = np.vstack((unscaled_samples, unscaled_centroid))
scaled_x = minmax_scale(unscaled_x, feature_range=(-0.1, 0.1))
samples, centroid = scaled_x[:-1, :], scaled_x[-1, :]
likelihood_values = likelihood_df_sorted.values
N, dim = samples.shape
distance_to_centroid = np.linalg.norm(samples - centroid, ord=2, axis=1)
x_order = np.argsort(distance_to_centroid)


# This function fits a quadratic model f(X) = X^TAX + BX + C to the likelihood
# The Hessian matrix of f(X) is 2A and its gradient is 2AX + B
def fit_quadratic_model(X, y):
    poly_reg = PolynomialFeatures(degree=2)
    quadratic_monomials = poly_reg.fit_transform(X)

    regressor = LinearRegression(
        fit_intercept=False
    )  # intercept is already accounted for by quadratic_monomials
    regressor.fit(quadratic_monomials, y)
    # The r2 score will be used to quantify the goodness of fit of the quadratic model
    r2_score = regressor.score(quadratic_monomials, y)

    # assemble the Hessian matrix and gradient
    H = np.zeros((dim, dim))
    g = np.zeros((dim,))
    regressor_coefs = regressor.coef_.flatten()
    for r, row in enumerate(poly_reg.powers_):
        c = regressor_coefs[r]
        if np.sum(row) == 2:
            (nonzeros,) = np.nonzero(row)
            if len(nonzeros) == 1:
                i = nonzeros[0]
                H[i, i] += c  # dfdxidxi = aii
                g[i] += 2 * c * centroid[i]  #
            elif len(nonzeros) == 2:
                i, j = nonzeros[0], nonzeros[1]
                H[i, j] += c / 2
                H[j, i] += c / 2
                g[i] += c * centroid[j]
                g[j] += c * centroid[i]
            else:
                print(f"Unexpected monomial powers: {row}")
        elif np.sum(row) == 1:
            (nonzeros,) = np.nonzero(row)
            if len(nonzeros) == 1:
                i = nonzeros[0]
                g[i] += c
            else:
                print(f"Unexpected monomial powers: {row}")
        else:  # degree 0 does not contribute to the Hessian
            continue
    return H, g, r2_score


H, gradient, r2_score = fit_quadratic_model(samples, likelihood_values)
eigen_decomposition = np.linalg.eig(H)
print(f"Hessian matrix:\n {H}")
print(f"Gradient:\n {gradient}")
print(f"Quadratic fit R2 score = {r2_score}")
h_df = pd.DataFrame(
    data=np.hstack(
        (
            np.reshape(eigen_decomposition.eigenvalues, (dim, 1)),
            eigen_decomposition.eigenvectors.T,
        )
    ),
    columns=["eigenValue"] + list(pivot_df_sorted.columns),
)
h_df = h_df.sort_values(by="eigenValue", ascending=True)
h_df

# Step 9 : Analyze the quality of the approximation

We fit the quadratic model on the `n` samples closest to the MLE, for increasing values of `n`.
Stable eigenvalues, a small gradient and a high R2 score indicate a reliable approximation.

In [ ]:
# How many points (minimum) do we need to fit a quadratic model?
poly_reg = PolynomialFeatures(degree=2)
quadratic_monomials = poly_reg.fit_transform(samples)
min_n = quadratic_monomials.shape[1]
if N < min_n:
    raise Exception(
        f"In dimension {dim}, one needs at least {min_n} points to fit a quadratic model, got {N} instead"
    )

# Compute and plot the Hessian eigen values, gradient and r2 score for each value of n
n_vector = np.arange(min_n, N + 1, 1)
eigenvalues = []
r2_scores = []
gradients = []
for n in n_vector:
    x = samples[x_order[:n], :]
    y = likelihood_values[x_order[:n]]
    H, gradient, r2_score = fit_quadratic_model(x, y)
    # H is symmetric: eigvalsh returns real, sorted eigenvalues
    eigenvalues.append(np.linalg.eigvalsh(H))
    r2_scores.append(r2_score)
    gradients.append(np.linalg.norm(gradient, ord=2))
eigenvalues = np.array(eigenvalues)
fig = go.Figure(
    go.Scatter(x=n_vector, y=eigenvalues[:, 0], name=f"lambda_1", mode="lines")
)
for k in range(1, dim):
    fig.add_scatter(
        x=n_vector, y=eigenvalues[:, k], mode="lines", name=f"lambda_{k+1}")
fig.update_layout(width=950, xaxis_title="n",
                  yaxis_title="Hessian matrix eigen values")
fig.show()

fig = go.Figure(go.Scatter(x=n_vector, y=gradients, mode="lines"))
fig.update_layout(width=950, xaxis_title="n", yaxis_title="norm2(gradient)")
fig.show()

fig = go.Figure(go.Scatter(x=n_vector, y=r2_scores, mode="lines"))
fig.update_layout(width=950, xaxis_title="n",
                  yaxis_title="quadratic fit r2 score")
fig.show()

## Next steps
- Tighten or widen the exploration with `CUSTOM_RANGE`, or increase `m` for a denser exploratory Vpop.
- Open the exploratory Trial in jinkō to inspect individual patients.
- Explore the [modeling cookbooks](https://github.com/novainsilico/jinko-api-cookbook/tree/main/cookbooks/modeling) for pre-calibration uncertainty quantification, calibration debugging and subsampling.